# Hi-EF Phase 1: Party-A emotion-label oracle

Train a validation-only oracle diagnostic that forecasts Party B from context clips I--II plus the ground-truth emotion label of Party A (clip III). The fixed five model seeds and two label-permutation controls are executed in one job. The test partition is never loaded.

Attach `ptrnghieu/hi-ef-features-v2` and the saved output of `ptrnghieu/hief-multiseed-validation`. Enable a T4 GPU and Internet.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/kaggle/working/hi-ef-materials')
FEATURES = Path('/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2')
OUTPUT = Path('/kaggle/working/phase1_label_oracle')

if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'experiments'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'experiments'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'experiments'], check=True)
else:
    subprocess.run([
        'git', 'clone', '--branch', 'experiments', '--single-branch',
        'https://github.com/ptrnghieu/hi-ef-materials.git', str(REPO)
    ], check=True)

assert (FEATURES / '01_00059.pt').exists(), 'Attach ptrnghieu/hi-ef-features-v2'
print('Commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
# Locate the frozen baseline summary by content, independently of Kaggle's input directory name.
import json

valid = []
for summary_path in Path('/kaggle/input').rglob('validation_matrix_summary.json'):
    try:
        candidate = json.loads(summary_path.read_text())
    except Exception:
        continue
    if (
        candidate.get('test_evaluated') is False
        and candidate.get('seeds') == [42, 123, 456, 789, 1024]
        and set(candidate.get('models', [])) == {'context', 'full'}
    ):
        valid.append(summary_path)

if len(valid) != 1:
    raise RuntimeError(
        'Expected exactly one attached hief-multiseed-validation output; '
        f'found {len(valid)} valid summaries: {valid}'
    )
BASELINE_SUMMARY = valid[0]
print('Baseline summary:', BASELINE_SUMMARY)

In [ ]:
command = [
    'python', str(REPO / 'experiments/run_label_oracle_matrix.py'),
    '--manifest', str(REPO / 'experiments/manifests/source_folder_split_seed42.csv'),
    '--features-dir', str(FEATURES),
    '--baseline-summary', str(BASELINE_SUMMARY),
    '--output-dir', str(OUTPUT),
    '--seeds', '42', '123', '456', '789', '1024',
    '--replacement-seed-start', '1701', '--replacement-replicates', '20',
    '--epochs', '50', '--batch-size', '32', '--workers', '2',
    '--learning-rate', '1e-4', '--weight-decay', '1e-5',
    '--patience', '8', '--d-model', '512',
    '--temporal-layers', '2', '--inter-layers', '2',
    '--dropout', '0.1', '--face-pooling', 'masked'
]
subprocess.run(command, check=True)

In [ ]:
import pandas as pd

summary = json.loads((OUTPUT / 'label_oracle_summary.json').read_text())
assert summary['test_evaluated'] is False
assert summary['model_seeds'] == [42, 123, 456, 789, 1024]
assert all(item['label_counts_preserved'] for item in summary['manifest_audit'])
display(pd.read_csv(OUTPUT / 'context_raw_label_comparison.csv'))
print(json.dumps({
    'comparison': summary['comparison'],
    'label_controls': summary['label_controls'],
}, indent=2))

Run only via **Save Version → Save & Run All**. Preserve the entire `/kaggle/working/phase1_label_oracle` directory. This is an oracle construct-validity diagnostic, not a deployable model: the ground-truth Party-A emotion label is intentionally supplied as input.